In [ ]:


import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score
from sklearn.utils.class_weight import compute_class_weight
from itertools import combinations
from collections import defaultdict

# ============================================================
# CONFIG
# ============================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_ROOT = "./preprocessing/processed_multicancer"
RESULTS_ROOT = "./research_results"
os.makedirs(RESULTS_ROOT, exist_ok=True)

GS_TYPES = [
    "GS-BRCA",
    "GS-GBM",
    "GS-COAD",
    "GS-LGG",
    "GS-OV"
]

OMICS = ["mRNA", "miRNA", "CNV", "Methy"]

N_SPLITS = 5
MAX_EPOCHS = 300
MIN_EPOCHS = 80
PATIENCE = 40
LR = 1e-3
WEIGHT_DECAY = 5e-4

TOP_K_GENES = 20

# ============================================================
# MODEL
# ============================================================

class StrongMLP(nn.Module):
    def __init__(self, in_dim, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, 1024)
        self.fc2 = nn.Linear(1024, 512)
        self.fc3 = nn.Linear(512, 128)
        self.fc4 = nn.Linear(128, num_classes)

        self.relu = nn.ReLU()
        self.drop = nn.Dropout(0.5)

    def forward(self, x):
        x = self.drop(self.relu(self.fc1(x)))
        x = self.drop(self.relu(self.fc2(x)))
        x = self.drop(self.relu(self.fc3(x)))
        return self.fc4(x)

# ============================================================
# TRAINING FUNCTION
# ============================================================

def train_on_split(X, y, tr, te):
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(X[tr])
    Xte = scaler.transform(X[te])

    Xtr_t = torch.tensor(Xtr, dtype=torch.float32).to(DEVICE)
    Xte_t = torch.tensor(Xte, dtype=torch.float32).to(DEVICE)
    ytr_t = torch.tensor(y[tr], dtype=torch.long).to(DEVICE)

    weights = compute_class_weight(
        class_weight="balanced",
        classes=np.unique(y[tr]),
        y=y[tr]
    )

    loss_fn = nn.CrossEntropyLoss(
        weight=torch.tensor(weights, dtype=torch.float32).to(DEVICE)
    )

    model = StrongMLP(Xtr.shape[1], len(np.unique(y))).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    best_prec = 0.0
    best_state = model.state_dict()
    wait = 0

    for epoch in range(MAX_EPOCHS):
        model.train()
        opt.zero_grad()
        loss_fn(model(Xtr_t), ytr_t).backward()
        opt.step()

        model.eval()
        with torch.no_grad():
            preds = model(Xte_t).argmax(1).cpu().numpy()
            prec = precision_score(
                y[te], preds, average="weighted", zero_division=0
            )

        if prec > best_prec:
            best_prec = prec
            best_state = model.state_dict()
            wait = 0
        else:
            wait += 1

        if epoch >= MIN_EPOCHS and wait >= PATIENCE:
            break

    model.load_state_dict(best_state)
    return best_prec, model

# ============================================================
# MAIN PIPELINE
# ============================================================

for cancer in GS_TYPES:
    print(f"\n==================== {cancer} ====================")

    DATA_DIR = f"{DATA_ROOT}/{cancer}"
    OUT_DIR = f"{RESULTS_ROOT}/{cancer}"
    os.makedirs(OUT_DIR, exist_ok=True)

    try:
        X = {m: np.load(f"{DATA_DIR}/{m}_processed.npy") for m in OMICS}
        y = np.load(f"{DATA_DIR}/labels.npy")
        feature_names = {
            m: json.load(open(f"{DATA_DIR}/{m}_features.json"))
            for m in OMICS
        }
    except FileNotFoundError:
        print(f"Skipping {cancer} (missing files)")
        continue

    for m in OMICS:
        print(
            f"{m}: X dim = {X[m].shape[1]}, "
            f"features = {len(feature_names[m])}"
        )

    X_full = np.concatenate([X[m] for m in OMICS], axis=1)
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    # ========================================================
    # 1. PERFORMANCE + ABLATION
    # ========================================================

    full_scores = []
    ablation_scores = {m: [] for m in OMICS}
    weight_importance_per_fold = []

    for tr, te in skf.split(X_full, y):
        prec, model = train_on_split(X_full, y, tr, te)
        full_scores.append(prec)

        # ---- weight-based importance ----
        W = model.fc1.weight.detach().cpu().numpy()
        weight_importance_per_fold.append(np.linalg.norm(W, axis=0))

        # ---- modality ablation ----
        offset = 0
        for m in OMICS:
            d = X[m].shape[1]
            X_drop = X_full.copy()
            X_drop[:, offset:offset+d] = 0.0
            p_drop, _ = train_on_split(X_drop, y, tr, te)
            ablation_scores[m].append(p_drop)
            offset += d

    # ========================================================
    # 2. FEATURE IMPORTANCE TABLES
    # ========================================================

    mean_importance = np.mean(weight_importance_per_fold, axis=0)

    offset = 0
    gene_tables = {}

    for m in OMICS:
        d = X[m].shape[1]
        scores = mean_importance[offset:offset+d]
        genes = feature_names[m]

        min_len = min(len(scores), len(genes))
        scores = scores[:min_len]
        genes = genes[:min_len]

        df = pd.DataFrame({
            "Gene": genes,
            "Importance": scores
        }).sort_values("Importance", ascending=False)

        df.head(TOP_K_GENES).to_csv(
            f"{OUT_DIR}/top_{TOP_K_GENES}_{m}_genes.csv",
            index=False
        )

        gene_tables[m] = df
        offset += d

    # ========================================================
    # 3. FAEC (STABILITY)
    # ========================================================

    faec_counter = defaultdict(int)

    for imp in weight_importance_per_fold:
        offset = 0
        for m in OMICS:
            d = X[m].shape[1]
            scores = imp[offset:offset+d]
            genes = feature_names[m]

            min_len = min(len(scores), len(genes))
            scores = scores[:min_len]
            genes = genes[:min_len]

            top_idx = np.argsort(scores)[-TOP_K_GENES:]
            for idx in top_idx:
                faec_counter[(m, genes[idx])] += 1

            offset += d

    faec_df = pd.DataFrame([
        {"Omics": m, "Gene": g, "FAEC": c / N_SPLITS}
        for (m, g), c in faec_counter.items()
    ]).sort_values("FAEC", ascending=False)

    faec_df.to_csv(f"{OUT_DIR}/faec_gene_consistency.csv", index=False)

    # ========================================================
    # 4. CORI (REDUNDANCY)
    # ========================================================

    cori_records = []

    for m1, m2 in combinations(OMICS, 2):
        s1 = gene_tables[m1]["Importance"].values
        s2 = gene_tables[m2]["Importance"].values

        min_len = min(len(s1), len(s2))
        corr = np.corrcoef(s1[:min_len], s2[:min_len])[0, 1]

        cori_records.append({
            "Omics_A": m1,
            "Omics_B": m2,
            "CORI": abs(corr)
        })

    cori_df = pd.DataFrame(cori_records)
    cori_df.to_csv(f"{OUT_DIR}/cori_table.csv", index=False)

    # ========================================================
    # 5. SUMMARY
    # ========================================================

    summary = {
        "Mean_Precision": float(np.mean(full_scores)),
        "Std_Precision": float(np.std(full_scores))
    }

    pd.DataFrame([summary]).to_csv(
        f"{OUT_DIR}/performance_summary.csv",
        index=False
    )

    print(
        f"{cancer} PRECISION: "
        f"{summary['Mean_Precision']:.4f} ± {summary['Std_Precision']:.4f}"
    )

print("\nPipeline completed successfully on CPU.")



==================== GS-BRCA ====================
mRNA: X dim = 5000, features = 5000
miRNA: X dim = 200, features = 366
CNV: X dim = 5000, features = 5000
Methy: X dim = 5000, features = 5000
GS-BRCA PRECISION: 0.8724 ± 0.0242

==================== GS-GBM ====================
mRNA: X dim = 4511, features = 5000
miRNA: X dim = 129, features = 200
CNV: X dim = 5000, features = 5000
Methy: X dim = 5000, features = 5000
GS-GBM PRECISION: 0.8257 ± 0.0578

==================== GS-COAD ====================
mRNA: X dim = 5000, features = 5000
miRNA: X dim = 200, features = 200
CNV: X dim = 5000, features = 5000
Methy: X dim = 5000, features = 5000


/opt/miniconda3/envs/mlomics/lib/python3.9/site-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(


GS-COAD PRECISION: 0.8933 ± 0.0249

==================== GS-LGG ====================
mRNA: X dim = 3435, features = 5000
miRNA: X dim = 102, features = 200
CNV: X dim = 5000, features = 5000
Methy: X dim = 5000, features = 5000
